In [1]:
import pandas as pd


In [2]:
ad_df = pd.read_csv("ad_spend_data.csv")

web_df = pd.read_csv("web_analytics_log.csv")

crm_df = pd.read_csv("crm_conversion_data.csv")

In [3]:
print(ad_df.head())

         date        channel            campaign  amount_spent_usd  clicks  \
0  2024-01-01  Google_Search       GSearch_Brand           2098.57    3730   
1  2024-01-01  Google_Search     GSearch_Generic           1112.23    1310   
2  2024-01-01  Google_Search  GSearch_Competitor           2191.75     802   
3  2024-01-01  Meta_Facebook      FB_Retargeting           1303.84    2250   
4  2024-01-01  Meta_Facebook        FB_Lookalike            671.68     380   

   impressions    cpc  
0       100710  0.563  
1        20960  0.849  
2        12030  2.733  
3        33750  0.579  
4         4180  1.768  


In [4]:
print(web_df.head())

   session_id    user_id            timestamp        channel utm_source  \
0  SES_323717  USR_00001  2024-03-07 21:21:00  Meta_Facebook   facebook   
1  SES_352023  USR_00002  2024-03-27 20:39:00        Organic    organic   
2  SES_396567  USR_00002  2024-03-25 00:58:00  Meta_Facebook   facebook   
3  SES_638857  USR_00003  2024-03-13 02:06:00  Meta_Facebook   facebook   
4  SES_818699  USR_00003  2024-03-06 13:53:00         TikTok     tiktok   

  utm_medium    utm_campaign page_visited  session_duration_sec   device  
0        cpc    FB_Awareness     /pricing                   399  desktop  
1    organic     Organic_SEO        /blog                   279   mobile  
2        cpc  FB_Retargeting        /blog                   871  desktop  
3        cpc    FB_Lookalike     /product                   821   tablet  
4        cpc   TikTok_Video1        /home                   240   tablet  


In [5]:
print(crm_df.head())

  customer_id conversion_date  revenue_usd last_touch_channel  \
0   USR_01556      2024-03-17       106.69             TikTok   
1   USR_01940      2024-02-04       202.25      Google_Search   
2   USR_00475      2024-03-25       406.54      Google_Search   
3   USR_01552      2024-01-15       122.89              Email   
4   USR_01314      2024-02-07        22.03             TikTok   

  last_touch_campaign product_purchased  country  
0       TikTok_Video1        Basic Plan      USA  
1       GSearch_Brand        Basic Plan       UK  
2       GSearch_Brand          Pro Plan       UK  
3         Email_Promo   Enterprise Plan   Canada  
4        TikTok_Promo   Enterprise Plan  Germany  


In [6]:
ad_df.duplicated().sum()

np.int64(0)

In [7]:
web_df.duplicated().sum()

np.int64(0)

In [8]:
crm_df.duplicated().sum()

np.int64(0)

# Timestamp Validation

In [9]:
web_df["timestamp"] = pd.to_datetime(
    web_df["timestamp"],
    errors="coerce"
)

In [10]:
web_df["timestamp"].isna().sum()

np.int64(0)

In [11]:
print(web_df["timestamp"].min())
print(web_df["timestamp"].max())

2024-01-01 06:37:00
2024-03-28 19:08:00


In [12]:
crm_df["conversion_date"] = pd.to_datetime(
    crm_df["conversion_date"],
    errors="coerce"
)

In [13]:
crm_df["conversion_date"].isna().sum()

np.int64(0)

In [14]:
print(crm_df["conversion_date"].min())
print(crm_df["conversion_date"].max())

2024-01-05 00:00:00
2024-03-30 00:00:00


In [15]:
ad_df["date"] = pd.to_datetime(
    ad_df["date"],
    errors="coerce"
)

In [16]:
ad_df["date"].isna().sum()

np.int64(0)

In [17]:
print(ad_df["date"].min())
print(ad_df["date"].max())

2024-01-01 00:00:00
2024-03-30 00:00:00


In [18]:
web_df.duplicated(
    subset=["session_id"]
).sum()

np.int64(18)

In [19]:
web_df.duplicated(
    subset=["user_id","timestamp"]
).sum()

np.int64(1)

In [20]:
crm_df.duplicated(
    subset=["customer_id","conversion_date"]
).sum()

np.int64(0)

In [22]:
web_df = web_df.drop_duplicates(
    subset=["session_id"]
)

In [23]:
web_df["timestamp"] = pd.to_datetime(
    web_df["timestamp"],
    errors="coerce"
)

crm_df["conversion_date"] = pd.to_datetime(
    crm_df["conversion_date"],
    errors="coerce"
)

ad_df["date"] = pd.to_datetime(
    ad_df["date"],
    errors="coerce"
)

In [24]:
web_df["timestamp"].isna().sum()

crm_df["conversion_date"].isna().sum()

ad_df["date"].isna().sum()

np.int64(0)

In [25]:
print(web_df["timestamp"].min())
print(web_df["timestamp"].max())

2024-01-01 06:37:00
2024-03-28 19:08:00


In [26]:
last_sessions = (
    web_df
    .groupby("user_id")["timestamp"]
    .max()
    .reset_index()
)

merged = crm_df.merge(
    last_sessions,
    left_on="customer_id",
    right_on="user_id",
    how="left"
)

invalid = merged[
    merged["conversion_date"] <
    merged["timestamp"]
]

print(len(invalid))

190
